In [1]:
import pandas as pd
import numpy as np

DATA_DIR = "../../Data"  # adjust if your notebook's location differs

In [2]:
df1 = pd.read_csv(f"{DATA_DIR}/TMDB_all_movies.csv")
df2 = pd.read_csv(f"{DATA_DIR}/TMDB_movie_dataset_2.csv")
df3 = pd.read_csv(f"{DATA_DIR}/TrendingMovies.csv")


In [9]:
for name, df in [("TMDB_all_movies", df1), ("TMDB_movie_dataset_2", df2), ("TrendingMovies", df3)]:
    print(f"{name}: {df.shape}  {list(df.columns)}\n")


TMDB_all_movies: (1232116, 13)  ['id', 'title', 'overview', 'genres', 'keywords', 'cast', 'director', 'poster_path', 'release_date', 'vote_average', 'runtime', 'tagline', '_priority']

TMDB_movie_dataset_2: (1474875, 13)  ['id', 'title', 'overview', 'genres', 'keywords', 'cast', 'director', 'poster_path', 'release_date', 'vote_average', 'runtime', 'tagline', '_priority']

TrendingMovies: (9996, 13)  ['id', 'title', 'overview', 'genres', 'keywords', 'cast', 'director', 'poster_path', 'release_date', 'vote_average', 'runtime', 'tagline', '_priority']



In [5]:
for name, df in [("df1", df1), ("df2", df2), ("df3", df3)]:
    before = len(df)
    df["id"] = pd.to_numeric(df["id"], errors="coerce")
    df.dropna(subset=["id"], inplace=True)
    df["id"] = df["id"].astype("int64")
    if len(df) != before:
        print(f"{name}: dropped {before - len(df)} rows with a bad/missing id")

In [6]:
FINAL_COLUMNS = ["id", "title", "overview", "genres", "keywords", "cast",
                  "director", "poster_path", "release_date", "vote_average",
                  "runtime", "tagline"]

for df in (df1, df2, df3):
    for col in FINAL_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan

df1, df2, df3 = df1[FINAL_COLUMNS].copy(), df2[FINAL_COLUMNS].copy(), df3[FINAL_COLUMNS].copy()


In [7]:
df1["_priority"], df2["_priority"], df3["_priority"] = 1, 2, 3   # 1 = richest

stacked = pd.concat([df1, df2, df3], ignore_index=True)
stacked.sort_values(["id", "_priority"], inplace=True)
print(f"Rows stacked: {len(stacked)}  |  unique movie ids: {stacked['id'].nunique()}")


Rows stacked: 2716987  |  unique movie ids: 1473649


In [8]:
value_cols = [c for c in FINAL_COLUMNS if c != "id"]
final_df = stacked.groupby("id", as_index=False)[value_cols].first()
final_df = final_df.sort_values("id").reset_index(drop=True)

assert final_df["id"].is_unique
print(f"Final dataset: {final_df.shape[0]} unique movies, {final_df.shape[1]} columns")

Final dataset: 1473649 unique movies, 12 columns


In [13]:
# dupes = final_df[final_df.duplicated(subset=["title", "release_date"], keep=False)]
# if not dupes.empty:
#     print(f"Warning: {len(dupes)} rows share title + release_date under different ids:")
#     print(dupes[["id", "title", "release_date"]].sort_values("title").to_string(index=False))
# else:
#     print("No same title/release_date rows found under different ids — looks clean.")

# print("\nMissing values per column:")
# print(final_df.isna().sum())


In [ ]:
final_df.to_csv(f"{DATA_DIR}/Final_Dataset.csv", index=False)
print(f"Saved -> {DATA_DIR}/Final_Dataset.csv")

In [17]:
final_df.head(2)

,id,title,overview,genres,keywords,cast,director,poster_path,release_date,vote_average,runtime,tagline
0,2,Ariel,A Finnish man goes to the city to find a job a...,"Comedy, Drama, Romance, Crime","prison|underdog|helsinki, finland|factory work...","Turo Pajala, Susanna Haavisto, Matti Pellonpää...",Aki Kaurismäki,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg,1988-10-21,7.118,73.0,None
1,3,Shadows in Paradise,"Nikander, a rubbish collector and would-be ent...","Comedy, Drama, Romance","helsinki, finland|salesclerk|garbage","Matti Pellonpää, Kati Outinen, Sakari Kuosmane...",Aki Kaurismäki,/nj01hspawPof0mJmlgfjuLyJuRN.jpg,1986-10-17,7.300,74.0,None
